In [1]:
import torch
import torch.nn as nn
from Settings import TinyStoriesLM
from datetime import datetime
import random

from tokenizer import TinyStoriesTokenizer
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass, asdict
import numpy as np

print("cell ran")

cell ran


In [2]:
# ============= Hyper-parameters for training ============== #

@dataclass
class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 1
    vector_dim: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Pull a chunk of length block_size + 1 (data and target)
        chunk = self.data[idx : idx + self.block_size + 1]

        # Convert to torch tensors
        x = torch.from_numpy(chunk[:-1].astype(np.int64)) # Input
        y = torch.from_numpy(chunk[1:].astype(np.int64))  # Target (shifted by 1)

        return x, y 


class ProbeDataset(Dataset):
    def __init__(self, data, tensor_tags, index_range, limit=512, ignore_index = -100):
        self.data = data
        self.limit = limit
        self.index_range = list(index_range)
        self.tensor_tags = tensor_tags
        self.ignore_index = ignore_index
        #self.pos_to_id = pos_to_id

    def __len__(self):
        return len(self.index_range)

    def __getitem__(self, idx):
        idx_true = self.index_range[idx]
    
        # when we built tensors, we bactch (0, 512, 1024, 1536...)
        # To get the real position
        dataset_index = idx_true * self.limit

        # Extract binary file
        item = self.data[dataset_index]
        input_ids = item[0][:self.limit] 
        
        # from tensor tags
        labels = self.tensor_tags[idx_true][:self.limit].clone()
        labels[labels < 3] = self.ignore_index
        
        return {
            'input_ids': input_ids,
            'labels': labels
        }

print("probedataset ready")

probedataset ready


In [3]:
# We load the aligned data
tensores_tags_dataset = torch.load('tags_probes_5k_first_subtoken.pt')
POS_TAGS_VOCAB = [
    'PAD', 'SUBWORD', 'X', 'ADJ', 'ADP', 'ADV', 'AUX', 'CONJ', 
    'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 
    'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB'
]
tag2id = {tag: idx for idx, tag in enumerate(POS_TAGS_VOCAB)}
id2tag = {idx: tag for tag, idx in tag2id.items()}
ignore_index = -100
print(f"{len(tensores_tags_dataset)} tensors for Probing.")

5000 tensors for Probing.


In [4]:
import json

with open("multipos_dict.json", "r") as f:
    multipos_dict = json.load(f)

print("cell ran")

cell ran


In [5]:
#alto


In [15]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint_path = 'last_checkpoint_next_word.pt'

model = TinyStoriesLM.load(checkpoint_path, device=device).to(device)
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')

# We have to freeze all the model
for param in model.parameters():
    param.requires_grad = False

# We set our model i eval mode -> deactivate dropout  so the vectors are stable
# dropout turns off random neurons during training to avoid overfitting/memorizing 
model.eval() #same input = same output

activations = {}

def forward_hook(module, input, output): # hook function
    # 'output' - ouput from Transformer layer
    # module layer that was just executed
    # save in our dictionary
    # .detach() no gradient info / history - we just want numbers
    
    if isinstance(output, tuple):
        activations[module.name] = output[0].detach()
    else:
        activations[module.name] = output.detach()

hooks = []

# register each block (TransformerBlock)
for i, block in enumerate(model.transformers):
    block.name = f"layer_{i}" 
    handle = block.register_forward_hook(forward_hook)
    hooks.append(handle)


num_tags = len(tag2id)
input_dim = model.config.vector_dim
limit = model.config.block_size
num_layers = len(model.transformers)

# from 5,000 stories -> 4500 train and 500 validate
# data for training
train_subset_tags = tensores_tags_dataset[:4500]
val_subset_tags = tensores_tags_dataset[4500:]

# index_range to ensure aligment
train_ds = ProbeDataset(training_dataset, tensores_tags_dataset, index_range=range(0, 4500), limit=limit)
train_loader = DataLoader(train_ds, batch_size=model.config.batch_size, shuffle=True)

val_ds = ProbeDataset(training_dataset, tensores_tags_dataset, index_range=range(4500, 5000), limit=limit)
val_loader = DataLoader(val_ds, batch_size=model.config.batch_size, shuffle=False)

print(f"Subsets : training: {len(train_ds)} | validation: {len(val_ds)}")

# Layer for layer
for layer_to_probe in range(num_layers):
    layer_name = f"layer_{layer_to_probe}"
    print(f"\nTraining probe of: {layer_name}")

    # we create probe for each layer
    probe = nn.Linear(input_dim, num_tags).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)
    #criterion = nn.CrossEntropyLoss()

    # Training each probe
    for epoch in range(3):
        probe.train()
       
        for batch in train_loader:
            activations.clear()
            ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            with torch.no_grad():
                model(ids) # Forward pass del modelo para activar los hooks

            optimizer.zero_grad()
            logits = probe(activations[layer_name])
   
            loss = criterion(logits.view(-1, num_tags), labels.view(-1))
            loss.backward()
            optimizer.step()

        # we evaluate each epoch to see progress
        probe.eval()
        correct_tokens = 0
        total_tokens = 0

        # we evaluate ambiguity
        correct_ambiguous_tokens = 0
        total_ambiguous_tokens = 0
        ambiguous_tokens = {}
        
        with torch.no_grad():
            
            for v_batch in val_loader:
                v_ids = v_batch['input_ids'].to(device)
                v_labels = v_batch['labels'].to(device)
                
                model(v_ids)
                
                v_logits = probe(activations[layer_name])
                
                predictions = v_logits.argmax(dim=-1)

                # get probs with softmax and get top3
                v_probs = torch.softmax(v_logits, dim=-1)
                top_3_probs, top_3_idx = torch.topk(v_probs, k=3, dim=-1)  

                for b in range(v_ids.size(0)):
                    ids_list = v_ids[b].tolist()
                    tokens = tokenizer.decode_to_tokens(ids_list)
                    
                    for t in range(len(tokens)):
                        label_real = v_labels[b, t].item()
                        if label_real == ignore_index:
                            continue 
                    
                        current_word = tokens[t] 
            
                        # get top 3 tags and probs
                        top_3_lista = top_3_idx[b, t].tolist()
                        top_3_tags = [id2tag[idx] for idx in top_3_lista]
                        top_3_probs_list = top_3_probs[b, t].tolist()
                        percentages = [p * 100 for p in top_3_probs_list]

                        # save in dictionary and update variables to calculate accuracy
                        if current_word in multipos_dict:
                            pred_label = predictions[b, t].item()
                            total_ambiguous_tokens += 1
                            if pred_label == label_real:
                                correct_ambiguous_tokens += 1
                                
                            inicio_contexto = max(0, t - 10)
                            fin_contexto = min(len(ids_list),  t + 11)
                            texto_antes = tokenizer.decode(ids_list[inicio_contexto: t])
                            texto_despues = tokenizer.decode(ids_list[ t+1:fin_contexto])
                            sentence = f"Sentence: ...{texto_antes} *[{current_word.upper().strip()}]* {texto_despues}..."
                            ambiguous_tokens[current_word] = {
                                "real_tag": id2tag[label_real],
                                "pred_tag": id2tag[pred_label],
                                "top_3_tags": top_3_tags,
                                "percentages": percentages,
                                "sentence": sentence
                                
                            }
                
                # we don't want to evaluate tokens with ignore, just the one that have POS tags
                mask = (v_labels != ignore_index)
                correct_tokens += (predictions[mask] == v_labels[mask]).sum().item()
                total_tokens += mask.sum().item()                              
                 
        epoch_ambiguity_accuracy = (correct_ambiguous_tokens / total_ambiguous_tokens) * 100 if total_ambiguous_tokens > 0 else 0.0        
        epoch_accuracy = (correct_tokens / total_tokens) * 100 if total_tokens > 0 else 0.0
        
    word_samples = [' so',' playing' ,' back', ' it', ' he']
    print(f"General accuracy: {epoch_accuracy:.2f}%  | Ambiguity accuracy: {epoch_ambiguity_accuracy:.2f}%")
    
    for word in word_samples:
        info = ambiguous_tokens[word]
        
        print(f"\nWord: {repr(word)} | True Tag: {info['real_tag']} | Predicted Tag: {info['pred_tag']}")
        print(f"Top 1: {info['top_3_tags'][0]} ({info['percentages'][0]:.1f}%)")
        print(f"Top 2: {info['top_3_tags'][1]} ({info['percentages'][1]:.1f}%)")
        print(f"Top 3: {info['top_3_tags'][2]} ({info['percentages'][2]:.1f}%)")  
        print(f"{info['sentence']}")
               
    print(f"="*70) 
for handle in hooks:
    handle.remove()

# when training (ignore_index=-100) impo
print("cell ran")

Model loaded from last_checkpoint_next_word.pt (Epoch 0, iteration 130000)
Subsets : training: 4500 | validation: 500

Training probe of: layer_0
General accuracy: 96.78%  | ambiguity accuracy: 95.53%

Word: ' so' | True Tag: ADV | Predicted Tag: ADV
Top 1: ADV (84.8%)
Top 2: SCONJ (13.2%)
Top 3: ADP (0.5%)
Sentence: .... He had finally seen the glow he'd heard *[SO]*  much about. He thanked the fairy and ran home...

Word: ' playing' | True Tag: VERB | Predicted Tag: VERB
Top 1: VERB (97.5%)
Top 2: NOUN (2.3%)
Top 3: ADP (0.0%)
Sentence: ... folder. When she got there, she saw Sam *[PLAYING]*  with a ball. She gave him the folder and...

Word: ' back' | True Tag: ADV | Predicted Tag: ADV
Top 1: ADV (95.9%)
Top 2: VERB (1.5%)
Top 3: ADP (1.2%)
Sentence: ... They get lost. They don't know how to go *[BACK]* . "Tim, I'm hungry and tired!"...

Word: ' it' | True Tag: PRON | Predicted Tag: PRON
Top 1: PRON (99.9%)
Top 2: ADV (0.0%)
Top 3: NOUN (0.0%)
Sentence: ... a toy he wanted and asked